In [103]:
!pip install pandas sqlalchemy sqlacodegen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [sqlacodegen]


In [3]:
import pickle # pickel can remember the change in the file bt csv can't
import pandas as pd

In [7]:
data = [{"name": "Peace"}]

with open("users.pkl", 'wb') as f:
    pickle.dump(data, f) # pickle.dump

In [8]:
data_from_pkl = pickle.load(open('users.pkl', 'rb')) # pickle.loads -> pickle.load
print(data_from_pkl, type(data_from_pkl))

[{'name': 'Peace'}] <class 'list'>


In [9]:
df = pd.DataFrame(data)
df.head()

,name
0,Peace


In [21]:
df.to_pickle("df_pkl")
df.dtypes

name    str
dtype: object

In [22]:
df2 = pd.read_pickle("df_pkl")
df2.head()

,name
0,Peace


In [23]:
df2.dtypes

name    str
dtype: object

In [24]:
df.to_csv("my_df.csv", index=False)

In [46]:
from dataclasses import dataclass

@dataclass
class Author():
    name:str = "Dasha"
    genre:str = "sci-fi"
    year:int = None
    
# class Author():
#     name = "Dasha"
#     def __init__(self, name="Raw", *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.name = name

@dataclass
class Movie:
    name:str = 'Unknown'
    genre:str = 'Action'
    year:int = None

In [41]:
my_movie = Author()
my_movie.name = "Dasha Kravets"
my_movie.name

'Dasha Kravets'

In [42]:
data = [{
    "name": "Interstellar",
    "genre": "Sci-Fi"
},
{
    "name": "The Martian",
    "genre": "Sci-Fi"
},
{
    "name": "Arrival",
    "genre": "Sci-Fi"
}
]

In [45]:
df = pd.DataFrame(data)
df.head()

,name,genre
0,Interstellar,Sci-Fi
1,The Martian,Sci-Fi
2,Arrival,Sci-Fi


In [47]:
import sqlalchemy
from sqlalchemy import Column, Integer, String
from sqlalchemy import create_engine, Column, String, Integer
from sqlalchemy.orm import sessionmaker

In [48]:
engine = create_engine("sqlite:///app.db") # mysql, postgres

In [49]:
Session = sessionmaker(bind=engine) # just for the db to not be always active 
my_sess = Session()

In [52]:
#from sqlalchemy.ext.declarative import declarative_base # 4 our table to not be treated as an obj by python
from sqlalchemy.orm import declarative_base
Base = declarative_base() 

In [57]:
class Movie(Base): # table
    __tablename__ = "movies"
    
    id = Column(Integer, primary_key=True) # auto created for us
    name = Column(String) # 'Unknown' # col
    genre = Column(String) # 'Action' # col
    description = Column(String) # 'Action' # col
    year = Column(Integer, nullable=True) # None # col
    
    #__repr__ method defines how a Movie object appears when printed or viewed in the Python interpreter.
    def __repr__(self):
        return f"<Movie name = {self.name}>"
        
# saved instance = row

/var/folders/47/_j9368zd10z7q7_1hgrl5m0r0000gn/T/ipykernel_2339/136296623.py:1: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.Movie, and will be replaced in the string-lookup table.
  class Movie(Base): # table


InvalidRequestError: Table 'movies' is already defined for this MetaData instance.  Specify 'extend_existing=True' to redefine options and columns on an existing Table object.

In [59]:
# Add table(s) to database
Base.metadata.create_all(engine)

In [60]:
movie_obj = Movie(name='Interstellar', genre='Sci-Fi')
print(movie_obj , movie_obj.name)

<Movie name=Interstellar> Interstellar


In [61]:
movie_obj.id 

In [62]:
my_sess.add(movie_obj) # prepare to save
my_sess.commit() # save

In [63]:
print(movie_obj.id, movie_obj.description)

1 None


In [64]:
movie_obj2 = Movie(name='The Martian', genre='Sci-Fi')
print(movie_obj2.name, movie_obj2.id)

The Martian None


In [65]:
my_sess.add(movie_obj2)

In [66]:
movie_obj3 = Movie(name='Inception', genre='Sci-Fi')
print(movie_obj3.name, movie_obj3.id)

Inception None


In [67]:
my_sess.add(movie_obj3)

In [68]:
my_sess.commit()

In [69]:
print(movie_obj2.id, movie_obj3.id)

2 3


### CRUD in SQLAlchemy

**Create**  
**Retrieve**  
**Update**  
**Delete**

In [70]:
Session = sessionmaker(bind=engine)
session = Session()

In [90]:
# Create
movie = Movie(name='Guaridans of the Galaxy')
session.add(movie)
session.commit()

In [76]:
# Retreive

# Retrieve - Get 1 item by primary key
movie_a = session.get(Movie, 5)
if movie_a:
    print(f"ID: {movie_a.id}, Name: {movie_a.name}, Description: {movie_a.description}")
else:
    print("Movie with ID 5 not found")

Movie with ID 5 not found


In [77]:
# List
qs = session.query(Movie).all()
print(qs)

[<Movie name=Interstellar>, <Movie name=The Martian>, <Movie name=Inception>, <Movie name=Guaridans of the Galaxy>]


In [78]:
# List & Filter by Column Value
qs = session.query(Movie).filter_by(name='Guardians of the Galaxy').all()
qs

[]

In [79]:
# List & Filter by Column Value Containing Something
qs = session.query(Movie).filter(Movie.name.contains("Guaridans")).all()
qs

[<Movie name=Guaridans of the Galaxy>]

In [80]:
# List & Filter by Column Value Containing Something
my_query = input("What are you looking for?\n") or "Unknown"
qs = session.query(Movie).filter(Movie.name.contains(my_query)).all()
print(qs)

What are you looking for?
 the


[<Movie name=The Martian>, <Movie name=Guaridans of the Galaxy>]


In [94]:
# Update

movie_a = session.get(Movie, 5)
movie_a.description = "A dream within a dream"
print(movie_a.id, movie_a.name, movie_a.description)
session.commit()

5 Guaridans of the Galaxy A dream within a dream


In [96]:
movie_a = session.get(Movie, 5)
print(movie_a.description)

A dream within a dream


In [97]:
qs = session.query(Movie).filter(Movie.name.contains("Guaridans")).all()
for movie_obj in qs:
    movie_obj.name = "Guardians of the Galaxy"
session.commit()

In [98]:
qs = session.query(Movie).filter(Movie.name.contains("Guaridans")).all()
qs

[]

In [100]:
# Delete
movie_a = session.get(Movie, 5)
session.delete(movie_a)
session.commit()

In [102]:
session.flush() #method that synchronizes your in-memory objects with the database, 
                 #but does not permanently commit the transaction.

## SQLAlchemy & Pandas

In [104]:
!sqlacodegen sqlite:///app2.db

from typing import Optional

from sqlalchemy import Integer, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

class Base(DeclarativeBase):
    pass


class Movies(Base):
    __tablename__ = 'movies'

    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    name: Mapped[Optional[str]] = mapped_column(String)
    genre: Mapped[Optional[str]] = mapped_column(String)
    description: Mapped[Optional[str]] = mapped_column(String)
    year: Mapped[Optional[int]] = mapped_column(Integer)


In [105]:
engine = create_engine("sqlite:///app2.db")
Session = sessionmaker(bind=engine)
session = Session()

In [106]:
from typing import Optional

from sqlalchemy import Integer, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

class Base(DeclarativeBase):
    pass


class MoviesApp2(Base):
    __tablename__ = 'movies'

    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    name: Mapped[Optional[str]] = mapped_column(String)
    genre: Mapped[Optional[str]] = mapped_column(String)
    description: Mapped[Optional[str]] = mapped_column(String)
    year: Mapped[Optional[int]] = mapped_column(Integer)

    def __repr__(self):
        return f"<Movie name={self.name}>"

In [108]:
qs = session.query(MoviesApp2).all()
print(qs)

[<Movie name=The Martian>, <Movie name=Inception>, <Movie name=Guardians of the Galaxy>, <Movie name=Guardians of the Galaxy>, <Movie name=Guardians of the Galaxy>, <Movie name=Guardians of the Galaxy>, <Movie name=Guardians of the Galaxy>, <Movie name= Galaxy>, <Movie name= Galaxy>, <Movie name= Galaxy>, <Movie name=Guardians of the Galaxy>]


In [110]:
for old_obj in qs:
    movie_obj = Movie(name=old_obj.name, genre=old_obj.genre, year=old_obj.year)
    print(movie_obj.name)
    my_sess.add(movie_obj)
my_sess.commit()
qs = session.query(MoviesApp2).all()
print(qs)

The Martian
Inception
Guardians of the Galaxy
Guardians of the Galaxy
Guardians of the Galaxy
Guardians of the Galaxy
Guardians of the Galaxy
 Galaxy
 Galaxy
 Galaxy
Guardians of the Galaxy
[<Movie name=The Martian>, <Movie name=Inception>, <Movie name=Guardians of the Galaxy>, <Movie name=Guardians of the Galaxy>, <Movie name=Guardians of the Galaxy>, <Movie name=Guardians of the Galaxy>, <Movie name=Guardians of the Galaxy>, <Movie name= Galaxy>, <Movie name= Galaxy>, <Movie name= Galaxy>, <Movie name=Guardians of the Galaxy>]


In [113]:
old_engine = create_engine("sqlite:///app2.db")
old_df = pd.read_sql_table("movies", old_engine)
old_df.head(n=15)

,id,name,genre,description,year
0,2,The Martian,Sci-Fi,NaN,None
1,3,Inception,Sci-Fi,NaN,None
2,4,Guardians of the Galaxy,NaN,NaN,None
3,6,Guardians of the Galaxy,NaN,NaN,None
4,7,Guardians of the Galaxy,NaN,NaN,None
5,8,Guardians of the Galaxy,NaN,NaN,None
6,9,Guardians of the Galaxy,NaN,NaN,None
7,10,Galaxy,NaN,NaN,None
8,11,Galaxy,NaN,NaN,None
9,12,Galaxy,NaN,NaN,None


In [114]:
current_engine = create_engine("sqlite:///app.db")
current_df = pd.read_sql_table("movies", current_engine)
current_df.head()

,id,name,genre,description,year
0,2,The Martian,Sci-Fi,NaN,None
1,3,Inception,Sci-Fi,NaN,None
2,4,Guardians of the Galaxy,NaN,NaN,None
3,6,Guardians of the Galaxy,NaN,NaN,None
4,7,Guardians of the Galaxy,NaN,NaN,None


In [115]:
final_df = pd.concat([current_df, old_df])
final_df.head(n=20)

,id,name,genre,description,year
0,2,The Martian,Sci-Fi,NaN,None
1,3,Inception,Sci-Fi,NaN,None
2,4,Guardians of the Galaxy,NaN,NaN,None
3,6,Guardians of the Galaxy,NaN,NaN,None
4,7,Guardians of the Galaxy,NaN,NaN,None
5,8,Guardians of the Galaxy,NaN,NaN,None
6,9,Guardians of the Galaxy,NaN,NaN,None
7,10,Galaxy,NaN,NaN,None
8,11,Galaxy,NaN,NaN,None
9,12,Galaxy,NaN,NaN,None


In [116]:
final_df = final_df[['name', 'genre', 'description', 'year']] #remove IDs
final_df.reset_index(inplace=True, drop=True)
final_df.head(n=20)

,name,genre,description,year
0,The Martian,Sci-Fi,NaN,None
1,Inception,Sci-Fi,NaN,None
2,Guardians of the Galaxy,NaN,NaN,None
3,Guardians of the Galaxy,NaN,NaN,None
4,Guardians of the Galaxy,NaN,NaN,None
5,Guardians of the Galaxy,NaN,NaN,None
6,Guardians of the Galaxy,NaN,NaN,None
7,Galaxy,NaN,NaN,None
8,Galaxy,NaN,NaN,None
9,Galaxy,NaN,NaN,None


In [118]:
final_df.to_sql(
    "new_movies", #new table name
    current_engine,
    if_exists='replace',#if table exists ?
    dtype = {
        "name": String,
        "genre": String,
        "year": Integer,
        "description": String,
    }
)

44

In [119]:
! sqlacodegen sqlite:///app.db

from sqlalchemy import BigInteger, Column, Index, Integer, MetaData, String, Table

metadata = MetaData()


t_movies = Table(
    'movies', metadata,
    Column('index', BigInteger),
    Column('name', String),
    Column('genre', String),
    Column('description', String),
    Column('year', Integer),
    Index('ix_movies_index', 'index')
)


t_new_movies = Table(
    'new_movies', metadata,
    Column('index', BigInteger),
    Column('name', String),
    Column('genre', String),
    Column('description', String),
    Column('year', Integer),
    Index('ix_new_movies_index', 'index')
)


In [120]:
import sqlite3


connection = sqlite3.connect('app.db')
connection2 = create_engine("sqlite:///app.db")

df = pd.read_sql_query("SELECT * FROM movies", connection2)
df.head()

,index,name,genre,description,year
0,0,The Martian,Sci-Fi,None,None
1,1,Inception,Sci-Fi,None,None
2,2,Guardians of the Galaxy,NaN,None,None
3,3,Guardians of the Galaxy,NaN,None,None
4,4,Guardians of the Galaxy,NaN,None,None
